# Importar codigos

Se obtienen las funciones del codigo

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import subprocess
import math
import random



In [2]:
import import_ipynb
from Codigos import *
from Codigos_obj import calcular_ms_ssim_lpips

0.5701703950988658
0.10048144820330392
[22143  5667  1107   628   455   369   262   242   187   141   178   166
   163   180   235   235   235   292   290   570   938  1629  6665  1283
   211    90   106    70    74    47    79    62    54    53    46    57
    54    47    49    59    66    38    52    60    41    51    47    35
    37    35    36    44    41    30    31    32    40    30    36    36
    29    31    39    35    31    32    31    38    32    36    30    30
    50    33    29    36    29    40    32    41    36    45    34    41
    43    42    49    39    39    44    39    44    50    47    30    38
    32    37    41    40    42    34    47    40    44    63    47    98
   179   236   499   369   173   133    97   149   123    97   154   133
   127   112   129   130    95   113   106    98   124   110   102   121
   138   116   127   125   108   125   115    98   126   120    86   122
    91   113   102   105    84   117   132   133   118   109   113   109
   157   152

c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: c:\Users\santi\Jeremias\TrabajoGrado\Codigos\Funciones_Matematicas\Formulas_Matematicas\lib\site-packages\lpips\weights\v0.1\vgg.pth


# Funcion para obtener todos los valores cuantitativos de la imagen

Obtiene los valores seleccionados de la parte teorica

In [3]:
def recodificar_imagen(img_path, metodo, formato, calidad):
    import os
    import subprocess
    from PIL import Image

    nombre = os.path.splitext(os.path.basename(img_path))[0]
    out_dir = f"imagenes/{metodo}/{formato}"
    os.makedirs(out_dir, exist_ok=True)

    out_path = f"{out_dir}/{nombre}_{metodo}_q{calidad}.{formato}"

    # ── Forzar RGB en entrada ──
    with Image.open(img_path) as im:
        if im.mode != "RGB":
            rgb_dir = "imagenes/_rgb_temp"
            os.makedirs(rgb_dir, exist_ok=True)
            rgb_path = os.path.join(rgb_dir, os.path.basename(img_path))
            if im.mode == "P" and "transparency" in im.info:
                im = im.convert("RGBA")
            im.convert("RGB").save(rgb_path)
            img_path = rgb_path

    if metodo == "pillow":
        img = Image.open(img_path).convert("RGB")
        if formato == "jpg":
            img.save(out_path, "JPEG", quality=calidad)
        elif formato == "webp":
            img.save(out_path, "WEBP", quality=calidad)
        elif formato == "png":
            img.save(out_path, "PNG", compress_level=int((100 - calidad)/10))

    elif metodo == "ffmpeg":
        cmd = [
            "ffmpeg", "-y",
            "-i", img_path,
            "-vf", "format=rgb24",
            "-q:v", str(int((100 - calidad)/10 + 2)),
            out_path
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    elif metodo == "imagemagick":
        cmd = [
            "magick",
            img_path,
            "-colorspace", "sRGB",
            "-quality", str(calidad),
            out_path
        ]
        subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    else:
        raise ValueError("Metodo no soportado")

    # ── Verificar que se generó el archivo ──
    if not os.path.exists(out_path):
        raise RuntimeError(f"Recodificación fallida — no se generó: {out_path}")

    # ── Forzar RGB en output — todos los métodos ──
    with Image.open(out_path) as im_out:
        if im_out.mode != "RGB":
            im_out_rgb = im_out.convert("RGB")  # convertir con archivo cerrado
        else:
            im_out_rgb = None

    if im_out_rgb is not None:
        im_out_rgb.save(out_path)

    return out_path

In [4]:
def features_histograma(path, prefix):
    import numpy as np

    f = {}

    hist, hist_norm = obtener_histograma_intensidades(path)

    # guardar todo
    f[f"{prefix}_hist"] = hist.tolist()
    f[f"{prefix}_hist_norm"] = hist_norm.tolist()
    f[f"{prefix}_hist_bins"] = list(range(256))

    # derivados
    hist_norm_safe = hist_norm[hist_norm > 0]

    f[f"{prefix}_hist_entropy"] = -np.sum(hist_norm_safe * np.log2(hist_norm_safe))
    f[f"{prefix}_hist_std"] = np.std(hist_norm)
    f[f"{prefix}_hist_mean"] = np.mean(hist_norm)
    f[f"{prefix}_hist_min"] = np.min(hist_norm)
    f[f"{prefix}_hist_max"] = np.max(hist_norm)

    mu = np.mean(hist_norm)
    f[f"{prefix}_hist_skew"] = np.mean((hist_norm - mu)**3)
    f[f"{prefix}_hist_kurt"] = np.mean((hist_norm - mu)**4)

    return f

In [5]:
def features_generales(path, prefix):
    f = {}

    # básicas
    f[f"{prefix}_rango"] = obtener_rango_dinamico(path)
    f[f"{prefix}_std"] = obtener_desv_estandar_brillo(path)

    # contraste
    f[f"{prefix}_michelson"] = obtener_contraste_michelson(path)

    # frecuencia
    f[f"{prefix}_hf"] = obtener_energia_altas_frecuencias(path)
    f[f"{prefix}_lf"] = obtener_energia_bajas_frecuencias(path)
    f[f"{prefix}_hlfr"] = obtener_hlfr(path)

    # bordes multi escala
    f[f"{prefix}_edges_s1"] = obtener_densidad_bordes(path, sigma=1.0)
    f[f"{prefix}_edges_s2"] = obtener_densidad_bordes(path, sigma=2.0)
    f[f"{prefix}_edges_s3"] = obtener_densidad_bordes(path, sigma=3.0)

    # gradiente
    f[f"{prefix}_grad_mean"] = obtener_magnitud_media_gradiente(path)
    f[f"{prefix}_grad_std"] = obtener_desv_estandar_gradiente(path)

    # textura
    f[f"{prefix}_glcm_contrast"] = obtener_contraste_textura(path)
    f[f"{prefix}_glcm_hom"] = obtener_homogeneidad_textura(path)
    f[f"{prefix}_glcm_energy"] = obtener_energia_textura(path)
    f[f"{prefix}_glcm_corr"] = obtener_correlacion_glcm(path)

    # color
    f[f"{prefix}_sat_mean"] = obtener_media_saturacion(path)
    f[f"{prefix}_sat_var"] = obtener_varianza_saturacion(path)

    return f

In [6]:
def extraer_features(path, prefix):
    f = {}

    f.update(features_generales(path, prefix))
    f.update(features_histograma(path, prefix))

    return f

In [7]:
METODOS      = ["pillow", "ffmpeg", "imagemagick"]
FORMATOS     = ["jpg", "webp", "png"]
CARPETA_RAIZ = "imagenes"
CALIDADES_POOL = [10, 30, 50, 70, 90]
CALIDADES_POR_IMAGEN = 3
SEED = 42  # reproducible — mismas calidades siempre para la misma imagen
MIN_SIZE = 161  # mínimo requerido por MS-SSIM

In [8]:
def procesar_imagen(
    img_path,
    df,
    metodo="pillow",
    formato="jpg",
    calidad=50
):
    import pandas as pd

    if not isinstance(df, pd.DataFrame):
        raise TypeError("df debe ser un pandas.DataFrame")

    # ── Verificar tamaño mínimo y RGB en entrada ──
    MIN_SIZE = 161  # debe ser MAYOR que 160
    with Image.open(img_path) as im:
        w, h = im.size
        if w < MIN_SIZE or h < MIN_SIZE:
            escala      = MIN_SIZE / min(w, h)
            nuevo_w     = int(w * escala)
            nuevo_h     = int(h * escala)
            resize_dir  = "imagenes/_resize_temp"
            os.makedirs(resize_dir, exist_ok=True)
            resize_path = os.path.join(resize_dir, os.path.basename(img_path))
            im.convert("RGB").resize((nuevo_w, nuevo_h), Image.LANCZOS).save(resize_path)
            img_path = resize_path
            print(f"    ⚠ Imagen pequeña ({w}×{h}) → redimensionada a ({nuevo_w}×{nuevo_h})")

        elif im.mode != "RGB":
            rgb_dir  = "imagenes/_rgb_temp"
            os.makedirs(rgb_dir, exist_ok=True)
            rgb_path = os.path.join(rgb_dir, os.path.basename(img_path))
            if im.mode == "P" and "transparency" in im.info:
                im = im.convert("RGBA")
            im.convert("RGB").save(rgb_path)
            img_path = rgb_path

    # 1. recodificar
    out_path = recodificar_imagen(img_path, metodo, formato, calidad)

    # 2. features
    feats_orig = extraer_features(img_path, "orig")
    feats_comp = extraer_features(out_path, "comp")

    # 3. métrica objetivo
    metricas = calcular_ms_ssim_lpips(img_path, out_path)

    # 4. fila
    fila = {
        "img_original":     img_path,
        "img_recodificada": out_path,
        "metodo":           metodo,
        "formato":          formato,
        "calidad":          calidad,
        **feats_orig,
        **feats_comp,
        **metricas
    }

    return pd.concat([df, pd.DataFrame([fila])], ignore_index=True)

In [12]:
def procesar_dataset_completo(
    df: pd.DataFrame,
    formato_origen: str = "jpg",
    inicio: int = 0,        # índice de imagen donde empieza
    fin: int = None         # índice donde termina (None = hasta el final)
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()

    if formato_orig not in FORMATOS:
        raise ValueError(f"formato_origen debe ser uno de {FORMATOS}, recibido: '{formato_orig}'")

    carpeta = os.path.join(CARPETA_RAIZ, formato_orig)

    if not os.path.isdir(carpeta):
        raise FileNotFoundError(f"Carpeta no encontrada: {carpeta}")

    imagenes = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    if not imagenes:
        print(f"⚠ Sin imágenes en: {carpeta}")
        return df

    formatos_salida = [f for f in FORMATOS if f != formato_orig]
    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    mitad = math.ceil(len(imagenes) / 2)
    mitad_A = imagenes[:mitad]
    mitad_B = imagenes[mitad:]

    # ── Aplicar rango sobre la lista completa ──
    todas = list(zip(
        mitad_A + mitad_B,
        [fmt_A] * len(mitad_A) + [fmt_B] * len(mitad_B)
    ))

    fin_real = fin if fin is not None else len(todas)
    subset   = todas[inicio:fin_real]

    print(f"\n{'='*60}")
    print(f"📁 {carpeta}  ({len(imagenes)} imágenes total)")
    print(f"   Procesando [{inicio}:{fin_real}] → {len(subset)} imágenes")
    print(f"{'='*60}")

    # Avanzar el random hasta el índice correcto (reproducibilidad)
    for i in range(inicio):
        random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)

    for idx_global, (nombre_archivo, formato_sal) in enumerate(subset, start=inicio):

        img_path = os.path.join(carpeta, nombre_archivo)
        metodo   = METODOS[idx_global % len(METODOS)]

        calidades_img = random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)

        print(f"\n  🖼 [{idx_global}] {nombre_archivo}  →  {formato_sal}  [{metodo}]  calidades={calidades_img}")

        for calidad in calidades_img:
            try:
                df = procesar_imagen(
                    img_path = img_path,
                    df       = df,
                    metodo   = metodo,
                    formato  = formato_sal,
                    calidad  = calidad
                )
            except Exception as e:
                print(f"    ❌ q={calidad:2d}  {e}")
                continue

    print(f"\n✅ [{inicio}:{fin_real}] completado — filas acumuladas: {len(df)}")

    return df

In [24]:
# Celda 1 — se ejecuta, usa RAM, termina y libera
df = pd.DataFrame()
df = procesar_dataset_completo(df, formato_origen="jpg")
df.to_csv("csv/dataset_jpg.csv", index=False)
del df  # fuerza la liberación inmediata
print("✅ JPG listo")


📁 imagenes\jpg  (1503 imágenes)
   752 → webp  |  751 → png

  🖼 COCO_test2014_000000000080.jpg  →  webp  [pillow]  calidades=[10, 90, 50]

  🖼 COCO_test2014_000000000090.jpg  →  webp  [ffmpeg]  calidades=[50, 30, 10]

  🖼 COCO_test2014_000000000527.jpg  →  webp  [imagemagick]  calidades=[30, 10, 50]

  🖼 COCO_test2014_000000000535.jpg  →  webp  [pillow]  calidades=[90, 10, 50]

  🖼 COCO_test2014_000000000718.jpg  →  webp  [ffmpeg]  calidades=[70, 10, 90]

  🖼 COCO_test2014_000000001118.jpg  →  webp  [imagemagick]  calidades=[10, 30, 90]

  🖼 COCO_test2014_000000001459.jpg  →  webp  [pillow]  calidades=[90, 10, 50]

  🖼 COCO_test2014_000000002059.jpg  →  webp  [ffmpeg]  calidades=[30, 70, 10]

  🖼 COCO_test2014_000000002063.jpg  →  webp  [imagemagick]  calidades=[70, 50, 10]

  🖼 COCO_test2014_000000002677.jpg  →  webp  [pillow]  calidades=[30, 70, 90]

  🖼 COCO_test2014_000000003359.jpg  →  webp  [ffmpeg]  calidades=[50, 30, 10]

  🖼 COCO_test2014_000000003748.jpg  →  webp  [imagemag

In [25]:
# Celda 2 — arranca con RAM limpia
df = pd.read_csv("csv/dataset_jpg.csv")
df = procesar_dataset_completo(df, formato_origen="webp")
df.to_csv("csv/dataset_jpg_webp.csv", index=False)
del df
print("✅ WEBP listo")


📁 imagenes\webp  (1503 imágenes)
   752 → jpg  |  751 → png

  🖼 alternatives-to-onlyfans-2025-3819.webp  →  jpg  [pillow]  calidades=[10, 90, 50]

  🖼 colt-onlyfans-90-day-fiance-leaked-rumors.webp  →  jpg  [ffmpeg]  calidades=[50, 30, 10]

  🖼 does-onlyfans-need-id-9649.webp  →  jpg  [imagemagick]  calidades=[30, 10, 50]

  🖼 fanfox-onlyfans-earnings-kr-0559.webp  →  jpg  [pillow]  calidades=[90, 10, 50]

  🖼 kak-skachat-onlyfans-na-iphone-9649.webp  →  jpg  [ffmpeg]  calidades=[70, 10, 90]

  🖼 leaked-onlyfans-earnings-and-what-they-really-mean.webp  →  jpg  [imagemagick]  calidades=[10, 30, 90]

  🖼 lgbtq-onlyfans-management-7629.webp  →  jpg  [pillow]  calidades=[90, 10, 50]

  🖼 maor-luz-onlyfans-leaked-and-the-rise-of-leaked-onlyfans-culture.webp  →  jpg  [ffmpeg]  calidades=[30, 70, 10]

  🖼 megan-thee-stallion-onlyfans-belgium-0494.webp  →  jpg  [imagemagick]  calidades=[70, 50, 10]

  🖼 mistress-onlyfans-guide-3344.webp  →  jpg  [pillow]  calidades=[30, 70, 90]

  🖼 norsk-on

In [ ]:
# Bloque 1 — primeras 500
df = pd.read_csv("csv/dataset_jpg_webp.csv")
df = procesar_dataset_completo(df, formato_origen="png", inicio=0, fin=200)
df.to_csv("csv/dataset_png_0_200.csv", index=False)
del df


📁 imagenes\png  (1500 imágenes total)
   Procesando [0:500] → 500 imágenes

  🖼 [0] 00000000_512resized (2).png  →  jpg  [pillow]  calidades=[10, 90, 50]

  🖼 [1] 00000001_512resized (2).png  →  jpg  [ffmpeg]  calidades=[50, 30, 10]

  🖼 [2] 00000002_512resized.png  →  jpg  [imagemagick]  calidades=[30, 10, 50]

  🖼 [3] 00000003_300resized (2).png  →  jpg  [pillow]  calidades=[90, 10, 50]

  🖼 [4] 00000003_512resized (2).png  →  jpg  [ffmpeg]  calidades=[70, 10, 90]

  🖼 [5] 00000005_300resized.png  →  jpg  [imagemagick]  calidades=[10, 30, 90]

  🖼 [6] 00000007_224resized (2).png  →  jpg  [pillow]  calidades=[90, 10, 50]

  🖼 [7] 00000019_224resized (2).png  →  jpg  [ffmpeg]  calidades=[30, 70, 10]

  🖼 [8] 00000019_512resized (2).png  →  jpg  [imagemagick]  calidades=[70, 50, 10]

  🖼 [9] 00000019_512resized.png  →  jpg  [pillow]  calidades=[30, 70, 90]

  🖼 [10] 00000022_300resized.png  →  jpg  [ffmpeg]  calidades=[50, 30, 10]

  🖼 [11] 00000023_300resized.png  →  jpg  [imagemagick

KeyboardInterrupt: 

In [ ]:
# Bloque 2 — primeras 400
df = pd.read_csv("csv/dataset_png_0_200.csv")
df = procesar_dataset_completo(df, formato_origen="png", inicio=200, fin=400)
df.to_csv("csv/dataset_png_200_400.csv", index=False)
del df

In [ ]:
# Bloque 2 — siguientes 500
df = pd.read_csv("csv/dataset_png_200_400.csv")
df = procesar_dataset_completo(df, formato_origen="png", inicio=400, fin=600)
df.to_csv("csv/dataset_png_400_600.csv", index=False)
del df

In [ ]:
# Bloque 2 — siguientes 500
df = pd.read_csv("csv/dataset_png_200_400.csv")
df = procesar_dataset_completo(df, formato_origen="png", inicio=600, fin=800)
df.to_csv("csv/dataset_png_400_600.csv", index=False)
del df

In [ ]:
# Bloque 3 — últimas
df = pd.read_csv("csv/dataset_png_0_1000.csv")
df = procesar_dataset_completo(df, formato_origen="png", inicio=1000, fin=None)
df.to_csv("csv/dataset_png_completo.csv", index=False)
del df

In [2]:
df = pd.read_csv("csv/dataset_jpg_webp.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8988 entries, 0 to 8987
Data columns (total 63 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   img_original        8988 non-null   object 
 1   img_recodificada    8988 non-null   object 
 2   metodo              8988 non-null   object 
 3   formato             8988 non-null   object 
 4   calidad             8988 non-null   int64  
 5   orig_rango          8988 non-null   float64
 6   orig_std            8988 non-null   float64
 7   orig_michelson      8988 non-null   float64
 8   orig_hf             8988 non-null   float64
 9   orig_lf             8988 non-null   float64
 10  orig_hlfr           8988 non-null   float64
 11  orig_edges_s1       8988 non-null   float64
 12  orig_edges_s2       8988 non-null   float64
 13  orig_edges_s3       8988 non-null   float64
 14  orig_grad_mean      8988 non-null   float64
 15  orig_grad_std       8988 non-null   float64
 16  orig_g

In [38]:
import os

def obtener_imagenes_faltantes(carpeta, formato, df):

    print("\n===== DEBUG INICIO =====\n")

    # ---------------------------
    # 1. ARCHIVOS EN CARPETA
    # ---------------------------
    archivos_carpeta = [
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato.lower()}")
    ]

    print(f"📁 Total en carpeta ({carpeta}): {len(archivos_carpeta)}")
    print("Ejemplos carpeta:")
    for f in archivos_carpeta[:10]:
        print("  ", f)

    # ---------------------------
    # 2. DATAFRAME (nombres)
    # ---------------------------
    nombres_df = [
        os.path.basename(path)
        for path in df["img_original"]
        if isinstance(path, str)
    ]

    print(f"\n📊 Total en DataFrame: {len(nombres_df)}")
    print("Ejemplos DataFrame:")
    for f in nombres_df[:10]:
        print("  ", f)

    # ---------------------------
    # 3. CONJUNTOS
    # ---------------------------
    set_carpeta = set(archivos_carpeta)
    set_df = set(nombres_df)

    print(f"\n🧮 Únicos carpeta: {len(set_carpeta)}")
    print(f"🧮 Únicos dataframe: {len(set_df)}")

    # ---------------------------
    # 4. DIFERENCIA
    # ---------------------------
    faltantes = set_carpeta - set_df

    print(f"\n❌ FALTANTES: {len(faltantes)}")
    print("Ejemplos faltantes:")
    for f in list(faltantes)[:10]:
        print("  ", f)

    print("\n===== DEBUG FIN =====\n")

    return sorted(list(faltantes))

In [41]:
print("Imágenes JPG faltantes:", (obtener_imagenes_faltantes("Imagenes/jpg", "jpg", df)))


===== DEBUG INICIO =====

📁 Total en carpeta (Imagenes/jpg): 1503
Ejemplos carpeta:
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000535.jpg
   COCO_test2014_000000000718.jpg
   COCO_test2014_000000001118.jpg
   COCO_test2014_000000001459.jpg
   COCO_test2014_000000002059.jpg
   COCO_test2014_000000002063.jpg
   COCO_test2014_000000002677.jpg

📊 Total en DataFrame: 8988
Ejemplos DataFrame:
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000535.jpg

🧮 Únicos carpeta: 1503
🧮 Únicos dataframe: 2996

❌ FALTANTES: 10
Ejemplos faltantes:
   COCO_test2014_000000477338.jpg
   COCO_test2014_000000342901.jpg
   COCO_test2014_000000436876.jpg

In [40]:
print("Imágenes WEBP faltantes:", len(obtener_imagenes_faltantes("Imagenes/webp", "webp", df)))


===== DEBUG INICIO =====

📁 Total en carpeta (Imagenes/webp): 1503
Ejemplos carpeta:
   alternatives-to-onlyfans-2025-3819.webp
   colt-onlyfans-90-day-fiance-leaked-rumors.webp
   does-onlyfans-need-id-9649.webp
   fanfox-onlyfans-earnings-kr-0559.webp
   kak-skachat-onlyfans-na-iphone-9649.webp
   leaked-onlyfans-earnings-and-what-they-really-mean.webp
   lgbtq-onlyfans-management-7629.webp
   maor-luz-onlyfans-leaked-and-the-rise-of-leaked-onlyfans-culture.webp
   megan-thee-stallion-onlyfans-belgium-0494.webp
   mistress-onlyfans-guide-3344.webp

📊 Total en DataFrame: 8988
Ejemplos DataFrame:
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000535.jpg

🧮 Únicos carpeta: 1503
🧮 Únicos dataframe: 2996

❌ FAL

In [9]:
df.shape

(8988, 63)

In [11]:
df.head()

,img_original,img_recodificada,metodo,formato,calidad,orig_rango,orig_std,orig_michelson,orig_hf,orig_lf,...,comp_hist_std,comp_hist_mean,comp_hist_min,comp_hist_max,comp_hist_skew,comp_hist_kurt,ms_ssim,lpips_raw,lpips_norm,combinado
0,imagenes\jpg\COCO_test2014_000000000080.jpg,imagenes/pillow/webp/COCO_test2014_00000000008...,pillow,webp,10,0.997195,0.219470,0.880256,137202340.0,1.477768e+10,...,0.003013,0.003906,0.000004,0.009957,4.554472e-09,1.464811e-10,0.943150,0.349070,0.650930,0.797040
1,imagenes\jpg\COCO_test2014_000000000080.jpg,imagenes/pillow/webp/COCO_test2014_00000000008...,pillow,webp,90,0.997195,0.219470,0.880256,137202340.0,1.477768e+10,...,0.002790,0.003906,0.000000,0.009086,-3.230005e-10,1.063236e-10,0.996491,0.031497,0.968503,0.982497
2,imagenes\jpg\COCO_test2014_000000000080.jpg,imagenes/pillow/webp/COCO_test2014_00000000008...,pillow,webp,50,0.997195,0.219470,0.880256,137202340.0,1.477768e+10,...,0.002832,0.003906,0.000000,0.009865,1.265967e-09,1.168539e-10,0.982705,0.168511,0.831489,0.907097
3,imagenes\jpg\COCO_test2014_000000000090.jpg,imagenes/ffmpeg/webp/COCO_test2014_00000000009...,ffmpeg,webp,50,0.998036,0.181978,0.686133,100286920.0,3.036802e+10,...,0.003782,0.003906,0.000004,0.019529,9.139638e-08,1.233006e-09,0.889200,0.380521,0.619479,0.754339
4,imagenes\jpg\COCO_test2014_000000000090.jpg,imagenes/ffmpeg/webp/COCO_test2014_00000000009...,ffmpeg,webp,30,0.998036,0.181978,0.686133,100286920.0,3.036802e+10,...,0.003788,0.003906,0.000004,0.017781,9.093940e-08,1.194257e-09,0.892389,0.369839,0.630161,0.761275


In [42]:
def reprocesar_faltantes(
    df: pd.DataFrame,
    imagenes_faltantes: list,
    formato_origen: str = "jpg"
) -> pd.DataFrame:

    random.seed(SEED)

    formato_orig = formato_origen.lower().strip()
    carpeta      = os.path.join(CARPETA_RAIZ, formato_orig)
    formatos_salida = [f for f in FORMATOS if f != formato_orig]

    # Recuperar el formato de salida original de cada imagen
    # basado en su posición en el listado completo ordenado
    imagenes_todas = sorted([
        f for f in os.listdir(carpeta)
        if f.lower().endswith(f".{formato_orig}")
    ])

    mitad = math.ceil(len(imagenes_todas) / 2)
    mitad_A = imagenes_todas[:mitad]   # → fmt_A
    mitad_B = imagenes_todas[mitad:]   # → fmt_B

    fmt_A = formatos_salida[0]
    fmt_B = formatos_salida[1]

    print(f"\n{'='*60}")
    print(f"🔁 Reprocesando {len(imagenes_faltantes)} imágenes faltantes [{formato_orig}]")
    print(f"{'='*60}")

    for nombre_archivo in imagenes_faltantes:

        img_path = os.path.join(carpeta, nombre_archivo)

        if not os.path.exists(img_path):
            print(f"  ⚠ No encontrada en disco: {nombre_archivo}")
            continue

        # Recuperar índice y grupo original para mantener consistencia
        if nombre_archivo in mitad_A:
            idx        = mitad_A.index(nombre_archivo)
            formato_sal = fmt_A
        elif nombre_archivo in mitad_B:
            idx        = mitad_B.index(nombre_archivo)
            formato_sal = fmt_B
        else:
            print(f"  ⚠ No encontrada en ningún grupo: {nombre_archivo}")
            continue

        # Mismo método que le correspondía originalmente
        metodo = METODOS[idx % len(METODOS)]

        # Mismas calidades que le correspondían originalmente
        # random.seed + mismo orden garantiza reproducibilidad
        random.seed(SEED)
        for i, img in enumerate(mitad_A + mitad_B):
            calidades = random.sample(CALIDADES_POOL, CALIDADES_POR_IMAGEN)
            if img == nombre_archivo:
                calidades_img = calidades
                break

        print(f"\n  🖼 {nombre_archivo}  →  {formato_sal}  [{metodo}]  calidades={calidades_img}")

        for calidad in calidades_img:
            try:
                df = procesar_imagen(
                    img_path = img_path,
                    df       = df,
                    metodo   = metodo,
                    formato  = formato_sal,
                    calidad  = calidad
                )
                print(f"    ✅ q={calidad}")
            except Exception as e:
                print(f"    ❌ q={calidad}  {e}")
                continue

    print(f"\n✅ Reprocesamiento completado — filas acumuladas: {len(df)}")
    return df

In [88]:
faltantes_jpg = [
    'COCO_test2014_000000207934.jpg',
    'COCO_test2014_000000213956.jpg',
    'COCO_test2014_000000436876.jpg'
]

df = reprocesar_faltantes(df, faltantes_jpg, formato_origen="jpg")
df.to_csv("csv/dataset_jpg_webp.csv", index=False)
print(f"Total filas: {len(df)}")


🔁 Reprocesando 3 imágenes faltantes [jpg]

  🖼 COCO_test2014_000000207934.jpg  →  webp  [imagemagick]  calidades=[10, 90, 50]
    ⚠ Imagen pequeña (640×147) → redimensionada a (700×161)
    ✅ q=10
    ⚠ Imagen pequeña (640×147) → redimensionada a (700×161)
    ✅ q=90
    ⚠ Imagen pequeña (640×147) → redimensionada a (700×161)
    ✅ q=50

  🖼 COCO_test2014_000000213956.jpg  →  webp  [imagemagick]  calidades=[90, 70, 10]
    ⚠ Imagen pequeña (640×152) → redimensionada a (677×161)
    ✅ q=90
    ⚠ Imagen pequeña (640×152) → redimensionada a (677×161)
    ✅ q=70
    ⚠ Imagen pequeña (640×152) → redimensionada a (677×161)
    ✅ q=10

  🖼 COCO_test2014_000000436876.jpg  →  png  [pillow]  calidades=[50, 90, 30]
    ✅ q=50
    ✅ q=90
    ✅ q=30

✅ Reprocesamiento completado — filas acumuladas: 9018
Total filas: 9018


In [62]:
print("Imágenes JPG faltantes:", (obtener_imagenes_faltantes("Imagenes/jpg", "jpg", df)))


===== DEBUG INICIO =====

📁 Total en carpeta (Imagenes/jpg): 1503
Ejemplos carpeta:
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000535.jpg
   COCO_test2014_000000000718.jpg
   COCO_test2014_000000001118.jpg
   COCO_test2014_000000001459.jpg
   COCO_test2014_000000002059.jpg
   COCO_test2014_000000002063.jpg
   COCO_test2014_000000002677.jpg

📊 Total en DataFrame: 9009
Ejemplos DataFrame:
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000080.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000090.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000527.jpg
   COCO_test2014_000000000535.jpg

🧮 Únicos carpeta: 1503
🧮 Únicos dataframe: 3003

❌ FALTANTES: 3
Ejemplos faltantes:
   COCO_test2014_000000213956.jpg
   COCO_test2014_000000436876.jpg
   COCO_test2014_000000207934.jpg
